# exp063_ravaghi_vs_pixiux_lgbm_feature_parity_audit train

Strict raw-data replay audit for public Ravaghi LightGBM features versus Pixiux likelihood-PF LightGBM features.


## Contents

1. Setup and configuration
2. Raw competition input check
3. Public notebook feature replay and LGBM audit
4. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

from settings import ExperimentPaths, load_config, get_nested
from public_notebook_replay_audit import run_public_replay_audit

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Mode:", get_nested(config, "audit.mode"))
print("Ravaghi reference:", get_nested(config, "lineage.ravaghi_reference_notebook"))
print("Pixiux reference:", get_nested(config, "lineage.pixiux_reference_notebook"))
print("Variants:", [item["name"] for item in cfg_get(config, "model.variants", [])])


## 2. Raw competition input check


In [ ]:
train_dir = paths.train_data_dir
test_dir = paths.test_data_dir
sample_path = paths.sample_submission_path
train_files = sorted(train_dir.glob("*__horizontal_well.csv"))
test_files = sorted(test_dir.glob("*__horizontal_well.csv"))
print("Train dir:", train_dir, "wells=", len(train_files))
print("Test dir:", test_dir, "wells=", len(test_files))
print("Sample submission:", sample_path, "exists=", sample_path.exists())
if not train_files:
    raise FileNotFoundError(f"No train wells found under {train_dir}")
preview = pd.read_csv(train_files[0], nrows=5)
display(preview)


## 3. Public notebook feature replay and LGBM audit


In [ ]:
summary = run_public_replay_audit(
    data_dir=paths.raw_data_dir,
    output_dir=paths.artifacts_dir,
    n_jobs=cfg_get(config, "runtime.num_workers", 8),
    pf_seeds=cfg_get(config, "audit.pf_seeds", 128),
    pf_particles=cfg_get(config, "audit.pf_particles", 500),
    fast=bool(cfg_get(config, "audit.fast", False)),
    use_gpu=str(cfg_get(config, "audit.use_gpu", "auto")),
    max_wells=cfg_get(config, "audit.max_wells"),
    top_n_importance=int(cfg_get(config, "audit.feature_importance.top_n", 45)),
)
print(json.dumps(summary, indent=2))


## 4. Metrics and artifacts


In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / "ravaghi_vs_pixiux_public_replay_metrics.csv")
schema = pd.read_csv(paths.artifacts_dir / "ravaghi_vs_pixiux_public_replay_feature_schema.csv")
mean_importance = pd.read_csv(paths.artifacts_dir / "ravaghi_vs_pixiux_public_replay_feature_importance_mean.csv")
plot_path = paths.artifacts_dir / "ravaghi_vs_pixiux_public_replay_feature_importance_mean_top.png"

display(metrics.sort_values(["rmse_tvt", "variant", "model"]).head(40))
display(schema.groupby("variant").size().rename("feature_count").reset_index())
display(mean_importance.sort_values(["variant", "mean_importance"], ascending=[True, False]).groupby("variant").head(30))
model_dir = paths.artifacts_dir / "ravaghi_vs_pixiux_public_replay_lgb_models"
tracker_train_path = paths.artifacts_dir / "ravaghi_vs_pixiux_public_replay_tracker_features_train.csv.gz"
print("Saved Pixiux LGBM model dir:", model_dir, "exists=", model_dir.exists())
if model_dir.exists():
    print("Saved model files:", len(list(model_dir.glob("*.txt"))))
    print("Model manifest:", model_dir / "manifest.json", "exists=", (model_dir / "manifest.json").exists())
print("Reusable tracker train features:", tracker_train_path, "exists=", tracker_train_path.exists())
print("Mean feature importance plot:", plot_path, "exists=", plot_path.exists())
if plot_path.exists():
    display(Image(filename=str(plot_path)))
